Создание нейро-сотрудника на базе GPT для автоматизации консультаций, продаж и поддержки клиентов.     
Дата защиты 30.01.2025
Руководителя проекта Татьяна Некрасова
         


Создание нейро-сотрудника на базе GPT для автоматизации консультаций, продаж и поддержки клиентов


In [ ]:
#Установка библиотек
!pip install -q openai pandas numpy faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.7 MB/s eta 0:00:00


In [ ]:
#Импорт библиотек,Ввод API-ключа OpenAI
#импортируем системные модули (os, re), табличные и числовые библиотеки (pandas, numpy),
#FAISS для векторного поиска и OpenAI-клиент для эмбеддингов и генерации ответа, а getpass используется для безопасного ввода API-ключа.
import os, re, json
import pandas as pd
import numpy as np
import faiss
from openai import OpenAI
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Вставьте ваш OpenAI API Key: ")
client = OpenAI()
# os.environ словарь, который содержит все переменные окружения текущего процесса.
# getpass библиотека для секретного ввода ключа


Вставьте ваш OpenAI API Key: ··········


In [ ]:
#Загрузка данных
CSV_PATH = "/content/smartphones.csv"                 #csv таблица в текстовом виде

def load_data(path: str = CSV_PATH) -> pd.DataFrame:    #pd.DataFrame структура данных в библиотеке Pandas
                                                        # файл читается,превращается в таблицу,таблица сохраняется в переменную df
                                                        # переменная с именем df которой присвоен обьект

    # Проверка, что колонки на месте
    df = pd.read_csv(path)
    required_cols = {"title", "price_rub", "url"}
    missing = required_cols - set(df.columns)           #поиск недистающих колонок
    if missing:
        raise ValueError(f"В CSV не хватает колонок: {missing}. Есть: {list(df.columns)}")
    return df

df = load_data()
df.head()


,url,title,price_rub,source
0,https://www.holodilnik.ru/digital_tech/smartph...,Смартфон Samsung Galaxy S25 Ultra 5G 12+256GB ...,109990,holodilnik.ru
1,https://www.holodilnik.ru/digital_tech/smartph...,Смартфон Realme C67 (RMX3890) 256Gb 8Gb зеленый,19799,holodilnik.ru
2,https://www.holodilnik.ru/digital_tech/smartph...,Смартфон Samsung Galaxy S25 Ultra 5G 12+512GB ...,129990,holodilnik.ru
3,https://www.holodilnik.ru/digital_tech/smartph...,Смартфон Realme C75 (RMX3941) 256Gb 8Gb черный,15999,holodilnik.ru
4,https://www.holodilnik.ru/digital_tech/smartph...,Смартфон Tecno Spark 40 Pro 128Gb 8Gb черный,16990,holodilnik.ru


In [ ]:
#Создание векторной базы: эмбеддинги + FAISS индекс
EMB_MODEL = "text-embedding-3-small"

def row_to_text(row: pd.Series) -> str:
    # Текст, который будет “векторизоваться”
    # Можно расширять, если есть колонки (ram, storage, camera и т.д.)
    return f"{row['title']} | price: {row['price_rub']} | url: {row['url']}" #превращаем одну строку таблицы в один текстовый
                                                                             # “документ”,который потом можно векторизовать.
def embed_texts(texts, batch_size=128):      #эмбеддинги для текстов
    vectors = []                             #сюда мы будем складывать эмбеддинги.
    for i in range(0, len(texts), batch_size):          #берём тексты кусками, чтобы не отправлять слишком много за один запрос
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=EMB_MODEL, input=batch) #В качестве input передаётся батч текстов, каждый из которых
                                                                      #преобразуется в отдельный вектор, формируя матрицу эмбеддингов.
        vectors.extend([d.embedding for d in resp.data])  #вытаскиваем все эмбеддинги и складываем в vectors.


    return np.array(vectors, dtype="float32")

#Готовим тексты из таблицы
texts = [row_to_text(r) for _, r in df.iterrows()]     #Получаем список texts, где каждый элемент — текст “про один телефон”.
#Получаем эмбеддинги для всех текстов
X = embed_texts(texts, batch_size=128)                 #где х матрица эмбеддингов.
#Узнаём размерность вектора
dim = X.shape[1]

index = faiss.IndexFlatIP(dim)     # создаем индекс FAISS, в которую мы загрузим все вектора и будем искать “похожие”
#Нормализация для cosine similarity
faiss.normalize_L2(X)           #чтобы поиск был “по смыслу” правильнее.
index.add(X)    #Добавляем все векторы в индекс

print("FAISS index ready. vectors:", index.ntotal, "dim:", dim)
#превращаем каждую строку таблицы в текст, считаем эмбеддинги, нормализуем их, создаём FAISS-индекс и загружаем в него все
#вектора для быстрого поиска релевантных товаров по смыслу запроса.

FAISS index ready. vectors: 15 dim: 1536


In [ ]:
#Парсинг запроса (ищем бюджет)

def parse_query(query: str): #функция берет текст запроса и пытается вытащить бюджет
    q = query.lower()            # приводим к нижнему регистру
    budget = None
# "до 30000", "до 30 тыс", "бюджет 30к"
    m = re.search(r"(?:до|бюджет)\s*(\d+)\s*(k|к|тыс|тысяч)?", q)
    if m:
        budget = int(m.group(1))
        if m.group(2):
            budget *= 1000
        return q, budget
# "30000 руб", "30000₽"
    m = re.search(r"(\d{4,6})\s*(руб|₽)", q)
    if m:
        return q, int(m.group(1))
# "30 тыс", "30k" (если без "до")
    m = re.search(r"(\d+)\s*(k|к|тыс|тысяч)\b", q)
    if m:
        budget = int(m.group(1)) * 1000

    return q, budget

#возвращаем и нормализованный запрос q, и найденный бюджет
#Переменная m хранит результат регулярного поиска: если шаблон найден в m лежат найденные фрагменты текста, если нет — None.
#Эта функция берёт текст,делает его удобным,пытается способами найти бюджет,если нашла — превращает его в число
#Если не нашла — честно говорит: “бюджета нет”

#автоматический выбор сценария
def route_scenario(q: str) -> str:
    brands = ["samsung", "iphone", "айфон", "xiaomi", "redmi", "realme", "tecno", "honor", "huawei"]
    features = ["фото", "камера", "селфи", "видео", "игр", "батар", "экран", "памят", "соцсет", "инст", "tiktok"]

    if any(b in q for b in brands):
        return "BRAND"
    if any(f in q for f in features):
        return "FEATURES"
    return "BASIC"
    #Функция route_scenario — это автопереключатель режима:если увидели бренд → "BRAND"
     #Если увидели цель (фото/игры/соцсети) → "FEATURES"
     #Иначе → "BASIC"


In [ ]:
#выбираем кандидатов, выбираем наиболее смысловые близкие объекты к запросу пользователя.
def embed_query(q: str) -> np.ndarray:
    v = client.embeddings.create(model=EMB_MODEL, input=q).data[0].embedding
    v = np.array([v], dtype="float32")   #Превращаем список чисел в формат, с которым удобно работать
    faiss.normalize_L2(v)   # нормализуем вектор
    return v
#probe_k это кол-во ближайших векторов, кот Faiss возвращает на первом этапе поиска
#сначала попросим у FAISS 80 ближайших, чтобы потом выбрать лучшие 12 и,вернём таблицу кандидатов.
def faiss_retrieve(df: pd.DataFrame, q: str, need_n: int = 12, probe_k: int = 80) -> pd.DataFrame: #ищем похожие товары в FAISS
    qv = embed_query(q)
    D, I = index.search(qv, probe_k)      # ищем probe_k ближайших кандидатов по cosine similarity.
    cands = df.iloc[I[0]].copy()

    # Убираем возможные дубликаты по URL
    cands = cands.drop_duplicates(subset=["url"])

    # Берём ровно 12 наиболее релевантных по FAISS (а не по цене)
    return cands.head(need_n)
#запрос → эмбеддинг → FAISS поиск → top-80 → убрать дубли → взять top-12
#происходит отбор кандидатов:Превращаем текст запроса в эмбеддинг-вектор чисел, чтобы можно было искать по смыслу.
#FAISS по этому вектору находит 80 ближайших кандидатов по смыслу, убираем дубли и берём 12 лучших в контекст.

In [ ]:
PROMPT_SYSTEM = """
Ты — нейро-сотрудник "Нейро-аналитик конкурентов".
ТЫ ДОЛЖЕН выбирать модели ТОЛЬКО из списка КОНТЕКСТ.
Запрещено придумывать модели, характеристики, цены и ссылки.
Если подходящих моделей нет — скажи: "В контексте нет подходящих моделей."
Всегда выводи URL для каждой рекомендованной модели.Не придумывай ссылки.
ВАЖНО:
Если в контексте нет характеристик (камера, процессор, батарея),
запрещено утверждать их наличие.
Используй нейтральные формулировки:
"подходит по бюджету", "подходит для базовых задач",
"подходит для повседневного использования".


ФОРМАТ ОТВЕТА (строго, 3 карточки, без подпунктов):

Ответ 1
Название: ...
Цена: ... ₽
URL: ...
Почему подходит: ...

Ответ 2
Название: ...
Цена: ... ₽
URL: ...
Почему подходит: ...

Ответ 3
Название: ...
Цена: ... ₽
URL: ...
Почему подходит: ...
"""

def build_user_prompt(query: str, budget, context: str, scenario: str) -> str:
    # сценарии разные, но формат ответа один
    scenario_note = {
        "BRAND": "Сценарий: пользователь хочет конкретный бренд. Если брендовых моделей нет — предложи альтернативы из контекста и честно скажи, что бренда нет.",
        "FEATURES": "Сценарий: пользователь описал цели (фото/соцсети/игры). Подбирай под цели, но только из контекста.",
        "BASIC": "Сценарий: базовый подбор по бюджету.",
    }[scenario]

    return (
        f"Запрос: {query}\n"
        f"Бюджет: {budget if budget else 'не указан'}\n"
        f"{scenario_note}\n\n"
        f"КОНТЕКСТ (выбирай только из него):\n{context}\n\n"
        "Выведи ТОП-3 модели из контекста. Для каждой обязательно: точное название, точная цена, URL, короткое обоснование"

    )


In [ ]:
#Формирование контекста
def make_context_table(cands: pd.DataFrame, budget: int | None = None) -> str:
    rows = []
    for _, row in cands.iterrows():             #Цикл перебирает строки таблицы, работая с каждым кандидатом по отдельности
        ok = ""
        if budget is not None:
            ok = " в_бюджете" if row["price_rub"] <= budget else " выше_бюджета"
        rows.append(f"- {row['title']} — {row['price_rub']} ₽ ({row['url']}){ok}")
#формируем строку с названием, ценой и ссылкой на телефон и добавляем к ней пометку “в бюджете” или “выше бюджета”, после чего кладём эту строку в список контекста.
    return "\n".join(rows)



In [ ]:
BEST_CONFIG = {
    "model": "gpt-4o-mini",
    "temperature": 0.2,     # важно: ниже, чтобы не выдумывал
    "need_n": 12
}
def run_neuro_analyst(query: str, df_path: str = CSV_PATH):
    df = load_data(df_path)             #читаем CSV и получаем таблицу df со всеми телефонами
    q, budget = parse_query(query)      #парсим запрос и вытаскиваем бюджет
    scenario = route_scenario(q)        #Выбираем сценарий автоматически

    candidates = faiss_retrieve(df, q, need_n=BEST_CONFIG["need_n"], probe_k=80)   #выбираем кандидатов через FAISS
    context = make_context_table(candidates, budget=budget)                        #формируем контекст для GPT

    user_prompt = build_user_prompt(query, budget, context, scenario)        #собираем user prompt,запрос,бюджет,сценарий, контекст
                                                                            # список кандидатов
    #вызываем GPT
    answer = call_llm(
        system_prompt=PROMPT_SYSTEM,
        user_prompt=user_prompt,
        model=BEST_CONFIG["model"],
        temperature=BEST_CONFIG["temperature"],
    )
    return answer, scenario, candidates, budget

#функция реализует полный пайплайн: парсинг запроса, выбор сценария, семантический поиск кандидатов через FAISS и
# генерацию ответа GPT строго из найденного контекста


In [ ]:
query = "Подбери 3 телефона до 20000 рублей: для соцсетей и фото."
answer, scenario, candidates, budget = run_neuro_analyst(query)

print("SCENARIO:", scenario)
print("BUDGET:", budget)
print("\nCANDIDATES FOUND:", len(candidates))
print("\nANSWER:\n", answer)
print("\nTOP CANDIDATES:\n", candidates[["title","price_rub","url"]])


SCENARIO: FEATURES
BUDGET: 20000

CANDIDATES FOUND: 12

ANSWER:
 Ответ 1  
Название: Смартфон REDMI Note 14 8+256 GB Midnight Black  
Цена: 19990 ₽  
URL: https://www.holodilnik.ru/digital_tech/smartphones/redmi/note_14_8_256gb_midnight_black/  
Почему подходит: Подходит для соцсетей и фото благодаря хорошему объему памяти и производительности.

Ответ 2  
Название: Смартфон Tecno Spark 40 256Gb 8Gb черный  
Цена: 14990 ₽  
URL: https://www.holodilnik.ru/digital_tech/smartphones/tecno/spark_40_256gb_8gb_chernyy/  
Почему подходит: Подходит для повседневного использования и социальных сетей с достаточным объемом памяти.

Ответ 3  
Название: Смартфон Tecno Spark 40 256Gb 8Gb серый  
Цена: 14990 ₽  
URL: https://www.holodilnik.ru/digital_tech/smartphones/tecno/spark_40_256gb_8gb_seryy/  
Почему подходит: Подходит для базовых задач и соцсетей, обеспечивая необходимый объем памяти.

TOP CANDIDATES:
                                                 title  price_rub  \
6    Смартфон Realme Note